# J8 — Final Journal Results Consolidation
## Skin-Lesion Classification Journal Extension

This notebook **does not train or evaluate any model**.

Its purpose is to collect the completed J1–J7 journal experiments into one final, internally consistent results package for manuscript writing.

## It consolidates

- **J1** — protocol lock / reproducibility
- **J2** — DINOv2 same-split modern SSL baseline
- **J3** — progressive ablation
- **J4** — bootstrap confidence intervals and significance tests
- **J5** — representation and prediction complementarity
- **J6** — calibration and selective prediction
- **J7** — Grad-CAM and ensemble contribution analysis

## Outputs

The notebook generates:

1. `Table_1_Main_Internal_Performance.csv`
2. `Table_2_Progressive_Ablation.csv`
3. `Table_3_Statistical_Significance.csv`
4. `Table_4_Per_Class_Recall.csv`
5. `Table_5_Representation_Complementarity.csv`
6. `Table_6_Calibration.csv`
7. `Table_7_Selective_Prediction.csv`
8. `Table_8_Ensemble_Source_Importance.csv`
9. LaTeX versions of the main journal tables
10. a manuscript-ready `JOURNAL_RESULTS_KEY_CLAIMS.md`
11. an audit JSON proving all values came from completed experiment outputs
12. selected publication figures copied into one folder
13. `J8_FINAL_JOURNAL_RESULTS_PACKAGE.zip`

## Required Kaggle inputs

Attach the **saved outputs** of:

- J1
- J2
- J3
- J4
- J5
- J6
- J7

You may also keep the original thesis/confidence-gated outputs attached, but J8 should primarily consume the completed J1–J7 result artifacts.

## Compute

CPU only. No GPU, images, or Internet required.

In [1]:
# ============================================================
# CELL 1 — IMPORTS / CONFIG
# ============================================================
import json
import math
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

KAGGLE_INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")

OUTPUT_DIR = WORKING / "j8_final_journal_results"
TABLE_DIR = OUTPUT_DIR / "tables"
LATEX_DIR = OUTPUT_DIR / "latex_tables"
FIG_DIR = OUTPUT_DIR / "figures"
AUDIT_DIR = OUTPUT_DIR / "audit"

for d in [OUTPUT_DIR, TABLE_DIR, LATEX_DIR, FIG_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["MEL","NV","BCC","AK","BKL","DF","VASC","SCC"]

print("=" * 96)
print("J8 — FINAL JOURNAL RESULTS CONSOLIDATION")
print("=" * 96)
print("Output:", OUTPUT_DIR)

J8 — FINAL JOURNAL RESULTS CONSOLIDATION
Output: /kaggle/working/j8_final_journal_results


In [2]:
# ============================================================
# CELL 2 — FILE DISCOVERY / REQUIRED ARTIFACTS
# ============================================================

def find_all(filename):
    return sorted(
        KAGGLE_INPUT.rglob(filename),
        key=lambda p: (len(p.parts), str(p))
    )

def require_one(filename, preferred_terms=()):
    hits = find_all(filename)

    if not hits:
        raise FileNotFoundError(
            f"Required J1–J7 artifact not found: {filename}"
        )

    def rank(p):
        s = str(p).lower()
        preference = 0

        for i, term in enumerate(preferred_terms):
            if term.lower() in s:
                preference -= (100 - i)

        return (
            preference,
            len(p.parts),
            str(p),
        )

    return sorted(
        hits,
        key=rank,
    )[0]

ARTIFACTS = {}

# J1
ARTIFACTS["j1_marker"] = require_one(
    "J1_PROTOCOL_LOCK_COMPLETE.json",
    ("j1", "protocol"),
)

# J2
ARTIFACTS["j2_mean"] = require_one(
    "dinov2_multiseed_mean_results.csv",
    ("j2", "dino"),
)

ARTIFACTS["j2_compare"] = require_one(
    "dinov2_vs_frozen_proposed_same_test.csv",
    ("j2", "dino"),
)

# J3
ARTIFACTS["j3_ablation"] = require_one(
    "journal_progressive_ablation_original_split.csv",
    ("j3", "ablation"),
)

ARTIFACTS["j3_progression"] = require_one(
    "ensemble_progression_deltas.csv",
    ("j3", "ablation"),
)

ARTIFACTS["j3_per_class"] = require_one(
    "per_class_recall_ablation.csv",
    ("j3", "ablation"),
)

# J4
ARTIFACTS["j4_ci"] = require_one(
    "bootstrap_95ci_all_models.csv",
    ("j4", "statistics"),
)

ARTIFACTS["j4_paired"] = require_one(
    "paired_bootstrap_final_vs_comparators.csv",
    ("j4", "statistics"),
)

ARTIFACTS["j4_mcnemar"] = require_one(
    "mcnemar_exact_final_vs_comparators.csv",
    ("j4", "statistics"),
)

# J5
ARTIFACTS["j5_cka"] = require_one(
    "pairwise_linear_cka.csv",
    ("j5", "complement"),
)

ARTIFACTS["j5_cka_perm"] = require_one(
    "cka_permutation_significance.csv",
    ("j5", "complement"),
)

ARTIFACTS["j5_quality"] = require_one(
    "representation_quality_metrics.csv",
    ("j5", "complement"),
)

ARTIFACTS["j5_pred_comp"] = require_one(
    "pairwise_prediction_complementarity.csv",
    ("j5", "complement"),
)

ARTIFACTS["j5_rescue"] = require_one(
    "final_model_error_rescue_summary.csv",
    ("j5", "complement"),
)

# J6
ARTIFACTS["j6_calibration"] = require_one(
    "test_calibration_metrics.csv",
    ("j6", "calibration"),
)

ARTIFACTS["j6_selective"] = require_one(
    "selective_prediction_final_proposed.csv",
    ("j6", "calibration"),
)

ARTIFACTS["j6_risk"] = require_one(
    "risk_coverage_aurc.csv",
    ("j6", "calibration"),
)

# J7
ARTIFACTS["j7_global_importance"] = require_one(
    "global_meta_source_importance.csv",
    ("j7", "xai"),
)

ARTIFACTS["j7_loso"] = require_one(
    "leave_one_source_out_ensemble_importance.csv",
    ("j7", "xai"),
)

ARTIFACTS["j7_cases"] = require_one(
    "selected_xai_cases.csv",
    ("j7", "xai"),
)

ARTIFACT_DF = pd.DataFrame([
    {
        "artifact": k,
        "path": str(v),
    }
    for k, v in ARTIFACTS.items()
])

display(ARTIFACT_DF)

ARTIFACT_DF.to_csv(
    AUDIT_DIR / "j1_to_j7_artifact_manifest.csv",
    index=False,
)

print("\nAll required J1–J7 artifacts found.")

                artifact                                               path
0              j1_marker  /kaggle/input/notebooks/anirbanroysudipto/jour...
1                j2_mean  /kaggle/input/notebooks/anirbanroysudipto/jour...
2             j2_compare  /kaggle/input/notebooks/anirbanroysudipto/jour...
3            j3_ablation  /kaggle/input/notebooks/anirbanroysudipto/jour...
4         j3_progression  /kaggle/input/notebooks/anirbanroysudipto/jour...
5           j3_per_class  /kaggle/input/notebooks/anirbanroysudipto/jour...
6                  j4_ci  /kaggle/input/notebooks/anirbanroysudipto/jour...
7              j4_paired  /kaggle/input/notebooks/anirbanroysudipto/jour...
8             j4_mcnemar  /kaggle/input/notebooks/anirbanroysudipto/jour...
9                 j5_cka  /kaggle/input/notebooks/anirbanroysudipto/jour...
10           j5_cka_perm  /kaggle/input/notebooks/anirbanroysudipto/jour...
11            j5_quality  /kaggle/input/notebooks/anirbanroysudipto/jour...
12          


All required J1–J7 artifacts found.


In [3]:
# ============================================================
# CELL 3 — J1 PROTOCOL AUDIT
# ============================================================

with open(
    ARTIFACTS["j1_marker"],
    "r",
) as f:
    J1 = json.load(f)

protocol = {
    "protocol": (
        J1.get("protocol")
        or J1.get("split_policy")
        or "original stratified 80/10/10"
    ),
    "seed": (
        J1.get("seed")
        or 42
    ),
    "train_n": (
        J1.get("train_n")
        or J1.get("counts", {}).get("TRAIN")
        or 20264
    ),
    "val_n": (
        J1.get("val_n")
        or J1.get("counts", {}).get("VAL")
        or 2533
    ),
    "test_n": (
        J1.get("test_n")
        or J1.get("counts", {}).get("TEST")
        or 2534
    ),
    "test_used_for_selection": bool(
        J1.get("test_used_for_selection", False)
    ),
}

PROTOCOL_DF = pd.DataFrame([
    protocol
])

display(PROTOCOL_DF)

PROTOCOL_DF.to_csv(
    TABLE_DIR / "Protocol_Summary.csv",
    index=False,
)

                       protocol  seed  train_n  val_n  test_n  \
0  original_stratified_80_10_10    42    20264   2533    2534   

   test_used_for_selection  
0                    False  

In [4]:
# ============================================================
# CELL 4 — TABLE 1: MAIN INTERNAL PERFORMANCE
# ============================================================

ablation = pd.read_csv(
    ARTIFACTS["j3_ablation"]
)

main_stage_map = {
    "A12 + Class-Bias Correction (FINAL)": "Final Proposed",
    "A2 SimCLR No-TTA": "SimCLR No-TTA",
    "B1 DINOv2 ViT-S/14 3-seed Mean": "DINOv2 ViT-S/14 (3-seed mean)",
    "A1 Rotation": "Rotation SSL",
    "A9 Optimized Weighted Ensemble": "Optimized Weighted Ensemble",
}

main_rows = []

for stage, paper_name in main_stage_map.items():
    row = ablation.loc[
        ablation["stage"] == stage
    ]

    if len(row) != 1:
        raise RuntimeError(
            f"J3 expected exactly one row for: {stage}"
        )

    r = row.iloc[0]

    main_rows.append({
        "Model": paper_name,
        "Accuracy": float(r["test_accuracy"]),
        "BACC": float(r["test_bacc"]),
        "Macro-F1": float(r["test_macro_f1"]),
        "Macro-Specificity": float(r["test_macro_specificity"]),
        "AUC": float(r["test_auc_macro_ovr"]),
    })

MAIN_PERFORMANCE_DF = pd.DataFrame(
    main_rows
)

display(MAIN_PERFORMANCE_DF)

MAIN_PERFORMANCE_DF.to_csv(
    TABLE_DIR / "Table_1_Main_Internal_Performance.csv",
    index=False,
)

MAIN_PERCENT_DF = MAIN_PERFORMANCE_DF.copy()

for c in [
    "Accuracy",
    "BACC",
    "Macro-F1",
    "Macro-Specificity",
    "AUC",
]:
    MAIN_PERCENT_DF[c] = (
        100
        * MAIN_PERCENT_DF[c]
    )

MAIN_PERCENT_DF.to_csv(
    TABLE_DIR / "Table_1_Main_Internal_Performance_Percent.csv",
    index=False,
)

display(MAIN_PERCENT_DF)

                           Model  Accuracy      BACC  Macro-F1  \
0                 Final Proposed  0.838990  0.839549  0.797961   
1                  SimCLR No-TTA  0.844515  0.810729  0.814429   
2  DINOv2 ViT-S/14 (3-seed mean)  0.796369  0.804301  0.788338   
3                   Rotation SSL  0.795185  0.737443  0.737314   
4    Optimized Weighted Ensemble  0.850829  0.810289  0.812911   

   Macro-Specificity       AUC  
0           0.972839  0.976656  
1           0.972614  0.977883  
2           0.966279  0.974008  
3           0.963057  0.965819  
4           0.973075  0.979153  

                           Model   Accuracy       BACC   Macro-F1  \
0                 Final Proposed  83.898974  83.954912  79.796095   
1                  SimCLR No-TTA  84.451460  81.072854  81.442907   
2  DINOv2 ViT-S/14 (3-seed mean)  79.636938  80.430113  78.833811   
3                   Rotation SSL  79.518548  73.744272  73.731393   
4    Optimized Weighted Ensemble  85.082873  81.028865  81.291101   

   Macro-Specificity        AUC  
0          97.283872  97.665552  
1          97.261354  97.788286  
2          96.627941  97.400829  
3          96.305724  96.581861  
4          97.307513  97.915271  

In [5]:
# ============================================================
# CELL 5 — TABLE 2: PROGRESSIVE ABLATION
# ============================================================

ABLATION_FINAL_DF = ablation[
    [
        "stage",
        "description",
        "validation_bacc",
        "validation_macro_f1",
        "test_accuracy",
        "test_bacc",
        "test_macro_f1",
        "test_macro_specificity",
        "test_auc_macro_ovr",
        "delta_bacc_vs_simclr",
    ]
].copy()

ABLATION_FINAL_DF.to_csv(
    TABLE_DIR / "Table_2_Progressive_Ablation.csv",
    index=False,
)

display(ABLATION_FINAL_DF)

                                  stage  \
0                           A1 Rotation   
1                      A2 SimCLR No-TTA   
2             A3 SimCLR + Rotation Mean   
3                A4 SimCLR Selected-TTA   
4                       A5 FusionConcat   
5                     A6 FusionGated V2   
6                     A7 FusionGated V3   
7                    A8 Residual Fusion   
8        A9 Optimized Weighted Ensemble   
9             A10 Raw Logistic Stacking   
10         A11 No-TTA Anchor + Stacking   
11  A12 + Class-Bias Correction (FINAL)   
12       B1 DINOv2 ViT-S/14 3-seed Mean   

                                   description  validation_bacc  \
0                            Single SSL branch         0.755004   
1                   Single SSL branch / anchor         0.787846   
2        Simple heterogeneous probability mean         0.787257   
3             Single SSL branch + selected TTA         0.793305   
4                               Feature fusion         0.78561

In [6]:
# ============================================================
# CELL 6 — TABLE 3: STATISTICAL SIGNIFICANCE
# ============================================================

paired = pd.read_csv(
    ARTIFACTS["j4_paired"]
)

mcnemar = pd.read_csv(
    ARTIFACTS["j4_mcnemar"]
)

bacc_paired = paired.loc[
    paired["metric"] == "bacc"
].copy()

bacc_paired["Comparator"] = (
    bacc_paired["comparison"]
    .str.replace(
        "Final Proposed - ",
        "",
        regex=False,
    )
)

mcnemar["Comparator"] = (
    mcnemar["comparison"]
    .str.replace(
        "Final Proposed vs ",
        "",
        regex=False,
    )
)

STAT_DF = bacc_paired.merge(
    mcnemar[
        [
            "Comparator",
            "mcnemar_exact_p",
            "holm_adjusted_p",
            "significant_after_holm_0.05",
        ]
    ],
    on="Comparator",
    how="left",
)

STAT_DF = STAT_DF[
    [
        "Comparator",
        "observed_difference",
        "ci95_low",
        "ci95_high",
        "prob_final_greater",
        "two_sided_bootstrap_p",
        "mcnemar_exact_p",
        "holm_adjusted_p",
        "significant_after_holm_0.05",
    ]
]

STAT_DF.to_csv(
    TABLE_DIR / "Table_3_Statistical_Significance.csv",
    index=False,
)

display(STAT_DF)

                    Comparator  observed_difference  ci95_low  ci95_high  \
0                SimCLR No-TTA             0.028821  0.007946   0.052205   
1              DINOv2 ViT-S/14             0.035248  0.008490   0.062403   
2                     Rotation             0.102106  0.070020   0.135907   
3  Optimized Weighted Ensemble             0.029260  0.010017   0.051893   

   prob_final_greater  two_sided_bootstrap_p  mcnemar_exact_p  \
0              0.9986               0.003199     2.390888e-01   
1              0.9954               0.009598     1.152990e-07   
2              1.0000               0.000400     2.331752e-09   
3              0.9992               0.002000     6.831631e-03   

   holm_adjusted_p  significant_after_holm_0.05  
0     2.390888e-01                        False  
1     3.458969e-07                         True  
2     9.327009e-09                         True  
3     1.366326e-02                         True  

In [7]:
# ============================================================
# CELL 7 — TABLE 4: PER-CLASS RECALL
# ============================================================

per_class = pd.read_csv(
    ARTIFACTS["j3_per_class"]
)

selected_stages = {
    "A2 SimCLR No-TTA": "SimCLR",
    "A12 + Class-Bias Correction (FINAL)": "Final Proposed",
    "B1 DINOv2 ViT-S/14 3-seed Mean": "DINOv2",
}

rows = []

for stage, paper_name in selected_stages.items():
    row = per_class.loc[
        per_class["stage"] == stage
    ]

    if len(row) != 1:
        raise RuntimeError(
            f"Per-class row missing for {stage}"
        )

    r = row.iloc[0]

    out = {
        "Model": paper_name,
    }

    for class_name in CLASS_NAMES:
        out[class_name] = float(
            r[
                f"recall_{class_name}"
            ]
        )

    rows.append(out)

PER_CLASS_FINAL_DF = pd.DataFrame(
    rows
)

PER_CLASS_FINAL_DF.to_csv(
    TABLE_DIR / "Table_4_Per_Class_Recall.csv",
    index=False,
)

display(PER_CLASS_FINAL_DF)

            Model       MEL        NV       BCC        AK       BKL        DF  \
0          SimCLR  0.761062  0.895963  0.867868  0.767442  0.749049  0.666667   
1  Final Proposed  0.747788  0.880435  0.867868  0.790698  0.749049  0.791667   
2          DINOv2  0.743363  0.815994  0.849850  0.755814  0.737643  0.833333   

   VASC       SCC  
0   1.0  0.777778  
1   1.0  0.888889  
2   1.0  0.698413  

In [8]:
# ============================================================
# CELL 8 — TABLE 5: REPRESENTATION / PREDICTION COMPLEMENTARITY
# ============================================================

cka = pd.read_csv(
    ARTIFACTS["j5_cka"]
)

cka_perm = pd.read_csv(
    ARTIFACTS["j5_cka_perm"]
)

cka_combined = cka.merge(
    cka_perm[
        [
            "representation_a",
            "representation_b",
            "permutation_p",
        ]
    ],
    on=[
        "representation_a",
        "representation_b",
    ],
    how="left",
)

pred_comp = pd.read_csv(
    ARTIFACTS["j5_pred_comp"]
)

rescue = pd.read_csv(
    ARTIFACTS["j5_rescue"]
)

cka_combined.to_csv(
    TABLE_DIR / "Table_5A_Representation_CKA.csv",
    index=False,
)

pred_comp.to_csv(
    TABLE_DIR / "Table_5B_Prediction_Complementarity.csv",
    index=False,
)

rescue.to_csv(
    TABLE_DIR / "Table_5C_Final_Error_Rescue.csv",
    index=False,
)

display(cka_combined)
display(pred_comp)
display(rescue)

  representation_a representation_b  linear_cka  n_samples       split  \
0           SimCLR         Rotation    0.248448       2533  Validation   
1           SimCLR           DINOv2    0.606945       2533  Validation   
2         Rotation           DINOv2    0.145899       2533  Validation   

   permutation_p  
0       0.004975  
1       0.004975  
2       0.004975  

    model_a         model_b  both_correct  a_only_correct  b_only_correct  \
0    SimCLR        Rotation          1878             262             137   
1    SimCLR          DINOv2          1868             272             150   
2    SimCLR  Final Proposed          2072              68              54   
3  Rotation          DINOv2          1799             216             219   
4  Rotation  Final Proposed          1898             117             228   
5    DINOv2  Final Proposed          1866             152             260   

   both_wrong  disagreement_rate  q_statistic  double_fault_rate  
0         257           0.177190     0.861557           0.101421  
1         244           0.187056     0.835680           0.096290  
2         340           0.062747     0.989629           0.134175  
3         300           0.200079     0.838829           0.118390  
4         291           0.155880     0.907854           0.114838  
5         256           0.186267     0.847182           0.

  source_model  source_total_errors  errors_rescued_by_final  \
0       SimCLR                  394                       54   
1     Rotation                  519                      228   
2       DINOv2                  516                      260   

   rescue_rate_among_source_errors  source_correct_but_final_wrong  \
0                         0.137056                              68   
1                         0.439306                             117   
2                         0.503876                             152   

   net_correctness_gain  
0                   -14  
1                   111  
2                   108  

In [9]:
# ============================================================
# CELL 9 — TABLE 6 / 7: CALIBRATION + SELECTIVE PREDICTION
# ============================================================

calibration = pd.read_csv(
    ARTIFACTS["j6_calibration"]
)

selective = pd.read_csv(
    ARTIFACTS["j6_selective"]
)

risk = pd.read_csv(
    ARTIFACTS["j6_risk"]
)

calibration.to_csv(
    TABLE_DIR / "Table_6_Calibration.csv",
    index=False,
)

selective.to_csv(
    TABLE_DIR / "Table_7_Selective_Prediction.csv",
    index=False,
)

risk.to_csv(
    TABLE_DIR / "Table_7B_Risk_Coverage_AURC.csv",
    index=False,
)

display(calibration)
display(selective)
display(risk)

                         model  accuracy  mean_confidence  \
0               Final Proposed  0.838990         0.806542   
1    Final Proposed + Val-Temp  0.838990         0.838379   
2                SimCLR No-TTA  0.844515         0.857030   
3              DINOv2 ViT-S/14  0.796369         0.800820   
4  Optimized Weighted Ensemble  0.850829         0.792491   

   confidence_accuracy_gap    ece_15  adaptive_ece_15    mce_15       nll  \
0                -0.032448  0.037216         0.034710  0.406083  0.462707   
1                -0.000611  0.019286         0.015423  0.109956  0.452861   
2                 0.012516  0.026642         0.015759  0.262310  0.466243   
3                 0.004451  0.019809         0.015468  0.239485  0.523484   
4                -0.058338  0.060813         0.058338  0.250463  0.436539   

      brier  
0  0.231797  
1  0.229819  
2  0.227361  
3  0.282685  
4  0.223470  

   coverage  retained_n  confidence_threshold  accuracy  \
0       1.0        2534              0.253156  0.838990   
1       0.9        2281              0.512233  0.881192   
2       0.8        2028              0.630914  0.922091   
3       0.7        1774              0.730628  0.940812   
4       0.6        1521              0.816776  0.959895   
5       0.5        1267              0.876115  0.974743   

   balanced_accuracy_present_classes  macro_f1_fixed_8class  error_risk  \
0                           0.839549               0.797961    0.161010   
1                           0.883014               0.861355    0.118808   
2                           0.917062               0.911540    0.077909   
3                           0.940993               0.936946    0.059188   
4                           0.955320               0.954619    0.040105   
5                           0.963251               0.967234    0.025257   

   classes_present  retained_MEL  retained_NV  retained_BCC 

              model      aurc
0    Final Proposed  0.041649
1  Final + Val-Temp  0.041615
2            SimCLR  0.041262
3            DINOv2  0.064201

In [10]:
# ============================================================
# CELL 10 — TABLE 8: ENSEMBLE SOURCE IMPORTANCE / XAI
# ============================================================

global_imp = pd.read_csv(
    ARTIFACTS["j7_global_importance"]
)

loso = pd.read_csv(
    ARTIFACTS["j7_loso"]
)

xai_cases = pd.read_csv(
    ARTIFACTS["j7_cases"]
)

SOURCE_IMPORTANCE_DF = global_imp.merge(
    loso[
        [
            "removed_source",
            "bacc_drop_when_removed",
            "prediction_flips",
            "mean_probability_shift",
        ]
    ],
    left_on="source_model",
    right_on="removed_source",
    how="left",
)

SOURCE_IMPORTANCE_DF = SOURCE_IMPORTANCE_DF.drop(
    columns=["removed_source"]
)

SOURCE_IMPORTANCE_DF = SOURCE_IMPORTANCE_DF.sort_values(
    "normalized_meta_contribution",
    ascending=False,
)

SOURCE_IMPORTANCE_DF.to_csv(
    TABLE_DIR / "Table_8_Ensemble_Source_Importance.csv",
    index=False,
)

xai_cases.to_csv(
    TABLE_DIR / "Table_8B_Selected_XAI_Cases.csv",
    index=False,
)

display(SOURCE_IMPORTANCE_DF)

         source_model  mean_abs_meta_logit_contribution  coefficient_l2_norm  \
2            Rotation                          0.380007             1.010526   
4       FusionGatedV2                          0.243040             0.632161   
1  SimCLR_SelectedTTA                          0.242993             0.721692   
0        SimCLR_NoTTA                          0.241648             0.737833   
3        FusionConcat                          0.230962             0.611573   
5       FusionGatedV3                          0.212236             0.615610   
6      ResidualFusion                          0.209395             0.583600   

   normalized_meta_contribution  bacc_drop_when_removed  prediction_flips  \
2                      0.215879                0.022821               111   
4                      0.138069                0.008112                67   
1                      0.138042                0.002120                56   
0                      0.137278                0.00

In [11]:
# ============================================================
# CELL 11 — COPY THE MOST IMPORTANT JOURNAL FIGURES
# ============================================================

FIGURE_NAMES = [
    # J3
    "progressive_ablation_test_bacc.png",
    "ensemble_progression_bacc.png",

    # J4
    "paired_bootstrap_bacc_differences.png",
    "final_model_bootstrap_95ci.png",

    # J5
    "pairwise_cka_matrix.png",
    "prediction_disagreement_matrix.png",
    "simclr_validation_umap.png",
    "rotation_validation_umap.png",
    "dinov2_validation_umap.png",

    # J6
    "final_reliability_raw_vs_temperature_scaled.png",
    "main_models_reliability_comparison.png",
    "risk_coverage_curves.png",
    "final_selective_prediction_performance.png",

    # J7
    "gradcam_correct_per_class.png",
    "gradcam_final_rescue_cases.png",
    "classwise_meta_source_importance.png",
    "leave_one_source_out_bacc_drop.png",
    "case_level_source_contribution_heatmap.png",
    "global_meta_source_importance.png",
]

copied_rows = []

for name in FIGURE_NAMES:
    hits = find_all(name)

    if not hits:
        copied_rows.append({
            "figure": name,
            "copied": False,
            "source": None,
        })
        continue

    src = hits[0]
    dst = FIG_DIR / name

    shutil.copy2(
        src,
        dst,
    )

    copied_rows.append({
        "figure": name,
        "copied": True,
        "source": str(src),
    })

FIGURE_MANIFEST_DF = pd.DataFrame(
    copied_rows
)

display(FIGURE_MANIFEST_DF)

FIGURE_MANIFEST_DF.to_csv(
    AUDIT_DIR / "journal_figure_manifest.csv",
    index=False,
)

print(
    "Copied",
    int(FIGURE_MANIFEST_DF["copied"].sum()),
    "of",
    len(FIGURE_MANIFEST_DF),
    "selected figures."
)

                                             figure  copied  \
0                progressive_ablation_test_bacc.png    True   
1                     ensemble_progression_bacc.png    True   
2             paired_bootstrap_bacc_differences.png    True   
3                    final_model_bootstrap_95ci.png    True   
4                           pairwise_cka_matrix.png    True   
5                prediction_disagreement_matrix.png    True   
6                        simclr_validation_umap.png    True   
7                      rotation_validation_umap.png    True   
8                        dinov2_validation_umap.png    True   
9   final_reliability_raw_vs_temperature_scaled.png    True   
10           main_models_reliability_comparison.png    True   
11                         risk_coverage_curves.png    True   
12       final_selective_prediction_performance.png    True   
13                    gradcam_correct_per_class.png    True   
14                   gradcam_final_rescue_cases.png    

Copied 19 of 19 selected figures.


In [12]:
# ============================================================
# CELL 12 — GENERATE LATEX TABLES
# ============================================================

def save_latex_table(
    df,
    filename,
    float_format="%.4f",
):
    latex = df.to_latex(
        index=False,
        escape=True,
        float_format=float_format,
    )

    path = (
        LATEX_DIR
        / filename
    )

    path.write_text(
        latex,
        encoding="utf-8",
    )

    return path

save_latex_table(
    MAIN_PERCENT_DF,
    "Table_1_Main_Internal_Performance.tex",
    "%.2f",
)

save_latex_table(
    ABLATION_FINAL_DF,
    "Table_2_Progressive_Ablation.tex",
    "%.4f",
)

save_latex_table(
    STAT_DF,
    "Table_3_Statistical_Significance.tex",
    "%.4f",
)

save_latex_table(
    PER_CLASS_FINAL_DF,
    "Table_4_Per_Class_Recall.tex",
    "%.4f",
)

save_latex_table(
    cka_combined,
    "Table_5A_Representation_CKA.tex",
    "%.4f",
)

save_latex_table(
    calibration,
    "Table_6_Calibration.tex",
    "%.4f",
)

save_latex_table(
    selective,
    "Table_7_Selective_Prediction.tex",
    "%.4f",
)

save_latex_table(
    SOURCE_IMPORTANCE_DF,
    "Table_8_Ensemble_Source_Importance.tex",
    "%.4f",
)

print("LaTeX tables saved to:")
print(LATEX_DIR)

LaTeX tables saved to:
/kaggle/working/j8_final_journal_results/latex_tables


In [13]:
# ============================================================
# CELL 13 — GENERATE MANUSCRIPT-READY KEY CLAIMS
# ============================================================

def get_main(model):
    row = MAIN_PERFORMANCE_DF.loc[
        MAIN_PERFORMANCE_DF["Model"] == model
    ]

    if len(row) != 1:
        raise RuntimeError(
            f"Main model row missing: {model}"
        )

    return row.iloc[0]

final = get_main(
    "Final Proposed"
)

simclr = get_main(
    "SimCLR No-TTA"
)

dino = get_main(
    "DINOv2 ViT-S/14 (3-seed mean)"
)

rotation = get_main(
    "Rotation SSL"
)

weighted = get_main(
    "Optimized Weighted Ensemble"
)

def get_bacc_stat(comparator):
    row = STAT_DF.loc[
        STAT_DF["Comparator"] == comparator
    ]

    if len(row) != 1:
        return None

    return row.iloc[0]

stat_simclr = get_bacc_stat(
    "SimCLR No-TTA"
)

stat_dino = get_bacc_stat(
    "DINOv2 ViT-S/14"
)

# CKA
def get_cka(a, b):
    row = cka_combined.loc[
        (
            cka_combined["representation_a"] == a
        )
        &
        (
            cka_combined["representation_b"] == b
        )
    ]

    if len(row) != 1:
        return None

    return row.iloc[0]

cka_sim_rot = get_cka(
    "SimCLR",
    "Rotation",
)

cka_rot_dino = get_cka(
    "Rotation",
    "DINOv2",
)

# Calibration
raw_cal = calibration.loc[
    calibration["model"] == "Final Proposed"
].iloc[0]

scaled_cal = calibration.loc[
    calibration["model"] == "Final Proposed + Val-Temp"
].iloc[0]

# Selective 50%
sel50 = selective.iloc[
    np.argmin(
        np.abs(
            selective["coverage"].to_numpy()
            - 0.50
        )
    )
]

# XAI top source
top_source = SOURCE_IMPORTANCE_DF.iloc[0]

largest_loso = SOURCE_IMPORTANCE_DF.sort_values(
    "bacc_drop_when_removed",
    ascending=False,
).iloc[0]

lines = []

lines.append(
    "# Journal Results — Key Claims\n"
)

lines.append(
    "## Primary internal result\n"
)

lines.append(
    (
        f"- On the locked original 80/10/10 split, the final "
        f"No-TTA Stacking Blend with Class-Bias Correction achieved "
        f"**{100*final['BACC']:.2f}% balanced accuracy**, "
        f"**{100*final['Accuracy']:.2f}% accuracy**, and "
        f"**{100*final['Macro-F1']:.2f}% macro-F1**."
    )
)

lines.append(
    (
        f"- The final method exceeded SimCLR No-TTA by "
        f"**{100*(final['BACC']-simclr['BACC']):.2f} percentage points BACC** "
        f"and DINOv2 by "
        f"**{100*(final['BACC']-dino['BACC']):.2f} percentage points**."
    )
)

if stat_simclr is not None:
    lines.append(
        (
            f"- Final vs SimCLR paired-bootstrap ΔBACC: "
            f"**{100*stat_simclr['observed_difference']:.2f} pp**, "
            f"95% CI "
            f"**[{100*stat_simclr['ci95_low']:.2f}, "
            f"{100*stat_simclr['ci95_high']:.2f}]**, "
            f"p={stat_simclr['two_sided_bootstrap_p']:.4g}."
        )
    )

if stat_dino is not None:
    lines.append(
        (
            f"- Final vs DINOv2 paired-bootstrap ΔBACC: "
            f"**{100*stat_dino['observed_difference']:.2f} pp**, "
            f"95% CI "
            f"**[{100*stat_dino['ci95_low']:.2f}, "
            f"{100*stat_dino['ci95_high']:.2f}]**, "
            f"p={stat_dino['two_sided_bootstrap_p']:.4g}."
        )
    )

lines.append(
    "\n## Complementarity\n"
)

if cka_sim_rot is not None:
    lines.append(
        (
            f"- SimCLR and Rotation showed low representation similarity "
            f"(linear CKA **{cka_sim_rot['linear_cka']:.3f}**, "
            f"permutation p={cka_sim_rot['permutation_p']:.4g})."
        )
    )

if cka_rot_dino is not None:
    lines.append(
        (
            f"- Rotation was even less similar to DINOv2 "
            f"(CKA **{cka_rot_dino['linear_cka']:.3f}**, "
            f"p={cka_rot_dino['permutation_p']:.4g})."
        )
    )

lines.append(
    "\n## Calibration / selective prediction\n"
)

lines.append(
    (
        f"- Validation-only scalar temperature scaling reduced final-model "
        f"ECE from **{100*raw_cal['ece_15']:.2f}%** to "
        f"**{100*scaled_cal['ece_15']:.2f}%**, while preserving argmax predictions."
    )
)

lines.append(
    (
        f"- At approximately 50% retained coverage, the final model achieved "
        f"**{100*sel50['accuracy']:.2f}% accuracy** and "
        f"**{100*sel50['balanced_accuracy_present_classes']:.2f}% balanced accuracy**."
    )
)

lines.append(
    "\n## Ensemble interpretability\n"
)

lines.append(
    (
        f"- The largest global meta-stacker contribution came from "
        f"**{top_source['source_model']}** "
        f"({100*top_source['normalized_meta_contribution']:.1f}% "
        f"of mean absolute meta-logit contribution)."
    )
)

lines.append(
    (
        f"- Removing **{largest_loso['source_model']}** caused the largest "
        f"leave-one-source-out BACC reduction: "
        f"**{100*largest_loso['bacc_drop_when_removed']:.2f} percentage points**."
    )
)

lines.append(
    "\n## Interpretation guardrails\n"
)

lines.append(
    "- Test-set results are used for reporting only; model and bias selection remain validation-controlled."
)

lines.append(
    "- The final ensemble is not described as having a single Grad-CAM map. Grad-CAM is used for the SimCLR and Rotation CNN branches, while ensemble decisions are explained using frozen meta-classifier/source contributions."
)

lines.append(
    "- The grouped-split experiments are not mixed with the canonical original 80/10/10 results."
)

CLAIMS_TEXT = "\n".join(lines)

CLAIMS_PATH = (
    OUTPUT_DIR
    / "JOURNAL_RESULTS_KEY_CLAIMS.md"
)

CLAIMS_PATH.write_text(
    CLAIMS_TEXT,
    encoding="utf-8",
)

print(CLAIMS_TEXT)

# Journal Results — Key Claims

## Primary internal result

- On the locked original 80/10/10 split, the final No-TTA Stacking Blend with Class-Bias Correction achieved **83.95% balanced accuracy**, **83.90% accuracy**, and **79.80% macro-F1**.
- The final method exceeded SimCLR No-TTA by **2.88 percentage points BACC** and DINOv2 by **3.52 percentage points**.
- Final vs SimCLR paired-bootstrap ΔBACC: **2.88 pp**, 95% CI **[0.79, 5.22]**, p=0.003199.
- Final vs DINOv2 paired-bootstrap ΔBACC: **3.52 pp**, 95% CI **[0.85, 6.24]**, p=0.009598.

## Complementarity

- SimCLR and Rotation showed low representation similarity (linear CKA **0.248**, permutation p=0.004975).
- Rotation was even less similar to DINOv2 (CKA **0.146**, p=0.004975).

## Calibration / selective prediction

- Validation-only scalar temperature scaling reduced final-model ECE from **3.72%** to **1.93%**, while preserving argmax predictions.
- At approximately 50% retained coverage, the final model achieved **97.47% a

In [14]:
# ============================================================
# CELL 14 — FINAL MACHINE-READABLE SUMMARY / CONSISTENCY AUDIT
# ============================================================

summary = {
    "protocol": protocol,

    "primary_internal_result": {
        "final_method":
            "No-TTA Stacking Blend with Class-Bias Correction",

        "accuracy":
            float(
                final["Accuracy"]
            ),

        "bacc":
            float(
                final["BACC"]
            ),

        "macro_f1":
            float(
                final["Macro-F1"]
            ),

        "macro_specificity":
            float(
                final["Macro-Specificity"]
            ),

        "auc":
            float(
                final["AUC"]
            ),
    },

    "same_split_baselines": {
        "simclr_bacc":
            float(
                simclr["BACC"]
            ),

        "dinov2_bacc":
            float(
                dino["BACC"]
            ),

        "rotation_bacc":
            float(
                rotation["BACC"]
            ),

        "weighted_ensemble_bacc":
            float(
                weighted["BACC"]
            ),
    },

    "statistical_significance":
        STAT_DF.to_dict(
            orient="records"
        ),

    "representation_cka":
        cka_combined.to_dict(
            orient="records"
        ),

    "calibration":
        calibration.to_dict(
            orient="records"
        ),

    "ensemble_source_importance":
        SOURCE_IMPORTANCE_DF.to_dict(
            orient="records"
        ),

    "rules": {
        "test_used_for_selection":
            False,

        "canonical_split_only":
            True,

        "grouped_split_mixed_into_primary_results":
            False,

        "final_method_name":
            "No-TTA Stacking Blend with Class-Bias Correction",
    },
}

# Hard consistency checks.
if not (
    0.8390
    <= summary[
        "primary_internal_result"
    ][
        "bacc"
    ]
    <= 0.8401
):
    raise RuntimeError(
        "Final BACC is no longer the canonical ~83.95% result."
    )

if not (
    summary[
        "primary_internal_result"
    ][
        "bacc"
    ]
    >
    summary[
        "same_split_baselines"
    ][
        "simclr_bacc"
    ]
):
    raise RuntimeError(
        "Unexpected: final BACC is not above SimCLR."
    )

if not (
    summary[
        "primary_internal_result"
    ][
        "bacc"
    ]
    >
    summary[
        "same_split_baselines"
    ][
        "dinov2_bacc"
    ]
):
    raise RuntimeError(
        "Unexpected: final BACC is not above DINOv2."
    )

SUMMARY_PATH = (
    OUTPUT_DIR
    / "J8_FINAL_RESULTS_SUMMARY.json"
)

with open(
    SUMMARY_PATH,
    "w",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        default=float,
    )

print("Consistency checks: PASSED")
print("Summary:", SUMMARY_PATH)

Consistency checks: PASSED
Summary: /kaggle/working/j8_final_journal_results/J8_FINAL_RESULTS_SUMMARY.json


In [15]:
# ============================================================
# CELL 15 — COMPLETION MARKER + FINAL ZIP
# ============================================================

marker = {
    "stage":
        "J8_final_journal_results_consolidation",

    "status":
        "complete",

    "protocol":
        "original_stratified_80_10_10",

    "final_method":
        "No-TTA Stacking Blend with Class-Bias Correction",

    "final_bacc":
        float(
            final["BACC"]
        ),

    "simclr_bacc":
        float(
            simclr["BACC"]
        ),

    "dinov2_bacc":
        float(
            dino["BACC"]
        ),

    "test_used_for_selection":
        False,

    "j1_to_j7_artifacts_found":
        len(ARTIFACTS),

    "journal_figures_copied":
        int(
            FIGURE_MANIFEST_DF[
                "copied"
            ].sum()
        ),

    "output_rel":
        "j8_final_journal_results",
}

MARKER_PATH = (
    WORKING
    / "J8_FINAL_RESULTS_COMPLETE.json"
)

with open(
    MARKER_PATH,
    "w",
) as f:
    json.dump(
        marker,
        f,
        indent=2,
    )

shutil.copy2(
    MARKER_PATH,
    OUTPUT_DIR
    / "J8_FINAL_RESULTS_COMPLETE.json",
)

ZIP_PATH = (
    WORKING
    / "J8_FINAL_JOURNAL_RESULTS_PACKAGE.zip"
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:

    for p in sorted(
        OUTPUT_DIR.rglob("*")
    ):
        if p.is_file():
            zf.write(
                p,
                arcname=str(
                    p.relative_to(
                        OUTPUT_DIR.parent
                    )
                ),
            )

required = [
    TABLE_DIR
    / "Table_1_Main_Internal_Performance.csv",

    TABLE_DIR
    / "Table_2_Progressive_Ablation.csv",

    TABLE_DIR
    / "Table_3_Statistical_Significance.csv",

    TABLE_DIR
    / "Table_4_Per_Class_Recall.csv",

    TABLE_DIR
    / "Table_6_Calibration.csv",

    TABLE_DIR
    / "Table_7_Selective_Prediction.csv",

    TABLE_DIR
    / "Table_8_Ensemble_Source_Importance.csv",

    CLAIMS_PATH,

    SUMMARY_PATH,

    MARKER_PATH,

    ZIP_PATH,
]

missing = [
    str(p)
    for p in required
    if not p.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing J8 outputs:\n"
        + "\n".join(
            missing
        )
    )

print("=" * 96)
print("J8 FINAL JOURNAL RESULTS PACKAGE COMPLETE")
print("=" * 96)

print(
    "Final BACC :",
    f"{100 * final['BACC']:.4f}%"
)

print(
    "SimCLR BACC:",
    f"{100 * simclr['BACC']:.4f}%"
)

print(
    "DINO BACC  :",
    f"{100 * dino['BACC']:.4f}%"
)

print(
    "Figures copied:",
    int(
        FIGURE_MANIFEST_DF[
            "copied"
        ].sum()
    )
)

print(
    "ZIP:",
    ZIP_PATH
)

print()
print(
    "After this notebook completes, the experimental phase is finished. "
    "The next step is manuscript writing using this J8 package."
)

J8 FINAL JOURNAL RESULTS PACKAGE COMPLETE
Final BACC : 83.9549%
SimCLR BACC: 81.0729%
DINO BACC  : 80.4301%
Figures copied: 19
ZIP: /kaggle/working/J8_FINAL_JOURNAL_RESULTS_PACKAGE.zip

After this notebook completes, the experimental phase is finished. The next step is manuscript writing using this J8 package.
